In [2]:
import pandas as pd

df = pd.read_csv("/Volumes/mac1/data/limb/limb_variants_with_genes.csv")

with open("/Volumes/mac1/data/limb/limb_variants.vcf", "w") as vcf:
    vcf.write("##fileformat=VCFv4.2\n")
    vcf.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
    for _, row in df.iterrows():
        chrom = row["CHROM"]
        pos = row["POS"]
        vid = row["ID"] if pd.notnull(row["ID"]) else "."
        ref = row["REF"]
        alt = row["ALT"]
        vcf.write(f"{chrom}\t{pos}\t{vid}\t{ref}\t{alt}\t.\t.\t.\n")


In [5]:
import requests
import pandas as pd

# API GRCh37
ENSEMBL_GRCH37 = "https://grch37.rest.ensembl.org/overlap/region/human"

def annotate_variant(chrom, pos):
    url = f"{ENSEMBL_GRCH37}/{chrom}:{pos}-{pos}?feature=gene"
    r = requests.get(url, headers={"Content-Type": "application/json"})
    if not r.ok:
        return None
    data = r.json()
    if len(data) == 0:
        return None
    
    # On récupère la première annotation
    ann = data[0]
    return {
        "gene_id": ann.get("gene_id"),
        "gene_name": ann.get("external_name"),
        "feature_type": ann.get("feature_type"),
        "biotype": ann.get("biotype")
    }

def coding_status(biotype):
    if biotype == "protein_coding":
        return "coding"
    return "noncoding"

# Charger ton fichier
df = pd.read_csv("/Volumes/mac1/data/limb/limb_variants_with_genes.csv")

annotations = []
for _, row in df.iterrows():
    ann = annotate_variant(row["CHROM"], row["POS"])
    if ann:
        ann["coding_status"] = coding_status(ann["biotype"])
    else:
        ann = {"gene_id": None, "gene_name": None, "feature_type": None,
               "biotype": None, "coding_status": None}
    annotations.append(ann)

annot_df = pd.DataFrame(annotations)
final = pd.concat([df, annot_df], axis=1)

final.to_csv("/Volumes/mac1/data/limb/limb_variants_annotated_python.csv", index=False)


In [13]:
import requests
import pandas as pd
import json

VEP_GRCH37 = "https://grch37.rest.ensembl.org/vep/human/region"

def annotate_vep(chrom, pos, ref, alt):
    variant = f"{chrom} {pos} . {ref} {alt}"
    payload = {"variants": [variant]}

    r = requests.post(
        VEP_GRCH37,
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload)
    )

    if not r.ok:
        return None

    data = r.json()
    if len(data) == 0:
        return None

    ann = data[0]

    # On récupère la première conséquence
    if "transcript_consequences" in ann:
        tc = ann["transcript_consequences"][0]
        return {
            "consequence": ",".join(tc.get("consequence_terms", [])),
            "impact": tc.get("impact"),
            "putative_gene": tc.get("gene_symbol")
        }

    return None

df = pd.read_csv("/Volumes/mac1/data/limb/limb_variants_annotated_python.csv")

annotations = []
for _, row in df.iterrows():
    ann = annotate_vep(row["CHROM"], row["POS"], row["REF"], row["ALT"])
    if ann:
        annotations.append(ann)
    else:
        annotations.append({"consequence": None, "impact": None, "putative_gene": None})

annot_df = pd.DataFrame(annotations)

final = pd.concat([df, annot_df], axis=1)

coding_terms = [
    "missense_variant", "synonymous_variant", "stop_gained",
    "stop_lost", "frameshift_variant", "inframe_insertion",
    "inframe_deletion", "start_lost"
]

def coding_status(cons):
    if cons is None:
        return None
    return "coding" if any(t in cons for t in coding_terms) else "noncoding"

final = final.reset_index(drop=True)

final["coding_status_functional"] = final["consequence"].apply(coding_status)

final.to_csv("/Volumes/mac1/data/limb/limb_variants_full_annotation_vep.csv", index=False)


In [14]:
final

,CHROM,POS,ID,REF,ALT,ES,SE,LP,AF,SS,...,GENE,gene_id,gene_name,feature_type,biotype,coding_status,consequence,impact,putative_gene,coding_status_functional
0,1,14605948,rs75914458,A,G,7.8193,1.7421,5.14442,0.004535,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
1,1,14614858,rs147770579,T,C,8.1951,1.8070,5.24003,0.004373,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
2,1,14616891,rs140845648,A,G,8.1970,1.8073,5.24048,0.004374,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
3,1,22204154,rs181769482,C,T,6.5060,1.2476,6.73542,0.008005,NaN,...,"('ENSG00000142798', 'HSPG2')",ENSG00000142798,HSPG2,gene,protein_coding,coding,intron_variant,MODIFIER,HSPG2,noncoding
4,1,22225373,rs141061782,G,A,3.8464,0.8159,5.61547,0.014600,NaN,...,"('ENSG00000142798', 'HSPG2')",ENSG00000142798,HSPG2,gene,protein_coding,coding,intron_variant,MODIFIER,HSPG2,noncoding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,X,150797626,rs780968876,C,G,74.0001,16.4531,5.16298,0.000213,NaN,...,"('ENSG00000239429', 'RP11-45D17.1')",ENSG00000239429,RP11-45D17.1,gene,pseudogene,noncoding,intron_variant,MODIFIER,PASD1,noncoding
116,X,150797738,rs747435727,C,T,74.0001,16.4531,5.16298,0.000213,NaN,...,"('ENSG00000239429', 'RP11-45D17.1')",ENSG00000239429,RP11-45D17.1,gene,pseudogene,noncoding,intron_variant,MODIFIER,PASD1,noncoding
117,X,150797855,rs149904168,G,T,74.0001,16.4531,5.16298,0.000213,NaN,...,"('ENSG00000239429', 'RP11-45D17.1')",ENSG00000239429,RP11-45D17.1,gene,pseudogene,noncoding,intron_variant,MODIFIER,PASD1,noncoding
118,X,150801527,rs189894061,A,T,74.0544,16.4645,5.16336,0.000213,NaN,...,"('ENSG00000239429', 'RP11-45D17.1')",ENSG00000239429,RP11-45D17.1,gene,pseudogene,noncoding,intron_variant,MODIFIER,PASD1,noncoding


In [35]:
final[final.putative_gene == 'PIK3C2A']

,CHROM,POS,ID,REF,ALT,ES,SE,LP,AF,SS,...,GENE,gene_id,gene_name,feature_type,biotype,coding_status,consequence,impact,putative_gene,coding_status_functional
84,11,17172053,rs748302455,G,C,23.9171,5.2694,5.24749,0.000839,NaN,...,"('ENSG00000011405', 'PIK3C2A')",ENSG00000011405,PIK3C2A,gene,protein_coding,coding,missense_variant,MODERATE,PIK3C2A,coding


In [20]:
import gseapy as gp
import pandas as pd

genes = ["PIK3C2A", "DNAJC27", "IGSF10"]

# GO Biological Process
go_bp = gp.enrichr(
    gene_list=genes,
    gene_sets="/Volumes/mac1/genes/gene_sets/GO_Biological_Process_2021.gmt",
    outdir=None,
    no_plot=True
)

# GO Molecular Function
go_mf = gp.enrichr(
    gene_list=genes,
    gene_sets="/Volumes/mac1/genes/gene_sets/GO_Molecular_Function_2021.gmt",
    outdir=None,
    no_plot=True
)

# Reactome
reactome = gp.enrichr(
    gene_list=genes,
    gene_sets="/Volumes/mac1/genes/gene_sets/Reactome_2022.gmt",
    outdir=None,
    no_plot=True
)

# KEGG
kegg = gp.enrichr(
    gene_list=genes,
    gene_sets="/Volumes/mac1/genes/gene_sets/KEGG_2021_Human.gmt",
    outdir=None,
    no_plot=True
)

go_bp_df = go_bp.results
go_mf_df = go_mf.results
reactome_df = reactome.results
kegg_df = kegg.results

go_bp_df.head()


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Odds Ratio,Combined Score,Genes
0,GO_Biological_Process_2021.gmt,ERBB signaling pathway (GO:0038127),1/66,0.013620,0.027458,131.949618,566.881845,PIK3C2A
1,GO_Biological_Process_2021.gmt,cellular protein localization (GO:0034613),1/329,0.066666,0.075343,25.829224,69.946911,DNAJC27
2,GO_Biological_Process_2021.gmt,cellular protein-containing complex assembly (...,1/168,0.034426,0.046400,51.232836,172.601249,PIK3C2A
3,GO_Biological_Process_2021.gmt,cellular response to insulin stimulus (GO:0032...,1/129,0.026505,0.039127,66.964202,243.107165,PIK3C2A
4,GO_Biological_Process_2021.gmt,clathrin coat assembly (GO:0048268),1/15,0.003106,0.016505,598.158621,3453.933068,PIK3C2A


In [22]:
import pandas as pd

def filter_gseapy(df, padj_threshold=0.05):
    """
    Filtre les résultats GSEApy enrichr sur la base du p-value ajusté.
    df : DataFrame GSEApy (ex: go_bp.results)
    padj_threshold : seuil de p-value ajusté
    """
    # Harmonisation du nom de colonne
    cols = df.columns.str.lower()
    df.columns = cols

    # GSEApy utilise "adjusted p-value"
    if "adjusted p-value" in cols:
        padj_col = "adjusted p-value"
    elif "adj_pval" in cols:
        padj_col = "adj_pval"
    else:
        raise ValueError("Aucune colonne de p-value ajustée trouvée dans le DataFrame.")

    # Filtrage
    filtered = df[df[padj_col] < padj_threshold].copy()

    # Tri par p-value ajusté
    filtered = filtered.sort_values(by=padj_col)

    return filtered

# Exemple d'utilisation :
go_bp_filtered = filter_gseapy(go_bp.results)
reactome_filtered = filter_gseapy(reactome.results)
kegg_filtered = filter_gseapy(kegg.results)

go_bp_filtered.to_csv("/Volumes/mac1/data/limb/GO_BP_filtered.csv", index=False)
reactome_filtered.to_csv("/Volumes/mac1/data/limb/Reactome_filtered.csv", index=False)
kegg_filtered.to_csv("/Volumes/mac1/data/limb/KEGG_filtered.csv", index=False)


go_bp_filtered.head()


,gene_set,term,overlap,p-value,adjusted p-value,odds ratio,combined score,genes
30,GO_Biological_Process_2021.gmt,vasoconstriction (GO:0042310),1/11,0.002279,0.016505,826.257143,5027.076195,PIK3C2A
27,GO_Biological_Process_2021.gmt,smooth muscle contraction (GO:0006939),1/14,0.002900,0.016505,642.511111,3754.320868,PIK3C2A
4,GO_Biological_Process_2021.gmt,clathrin coat assembly (GO:0048268),1/15,0.003106,0.016505,598.158621,3453.933068,PIK3C2A
20,GO_Biological_Process_2021.gmt,positive regulation of cell migration involved...,1/18,0.003727,0.016505,495.514286,2770.996159,PIK3C2A
17,GO_Biological_Process_2021.gmt,platelet-derived growth factor receptor signal...,1/16,0.003313,0.016505,559.529032,3194.802659,PIK3C2A


In [24]:
import pandas as pd

# Charger les résultats filtrés
go_bp = pd.read_csv("/Volumes/mac1/data/limb/GO_BP_filtered.csv")
kegg = pd.read_csv("/Volumes/mac1/data/limb/KEGG_filtered.csv")
reactome = pd.read_csv("/Volumes/mac1/data/limb/Reactome_filtered.csv")

# Ajouter une colonne "source" pour identifier l'origine
go_bp["source"] = "GO_BP"
kegg["source"] = "KEGG"
reactome["source"] = "Reactome"

# Harmoniser les noms de colonnes
for df in [go_bp, kegg, reactome]:
    df.columns = df.columns.str.lower().str.replace(" ", "_")

# Concaténer les trois tables
all_results = pd.concat([go_bp, kegg, reactome], ignore_index=True)

# Filtrer sur p-value ajustée < 0.05 (sécurité)
filtered = all_results[all_results["adjusted_p-value"] < 0.05].copy()

# Trier par p-value ajustée
filtered = filtered.sort_values(by="adjusted_p-value")

filtered.to_csv("/Volumes/mac1/data/limb/all_pathways_filtered.csv", index=False)

# Afficher les 10 pathways les plus significatifs
filtered.head(10)


,gene_set,term,overlap,p-value,adjusted_p-value,odds_ratio,combined_score,genes,source
27,Reactome_2022.gmt,Synthesis Of PIPs At Early Endosome Membrane R...,1/16,0.001526,0.006484,2026.935484,13145.483948,PIK3C2A,Reactome
28,Reactome_2022.gmt,Synthesis Of PIPs At Golgi Membrane R-HSA-1660514,1/17,0.001621,0.006484,1903.909091,12232.185019,PIK3C2A,Reactome
29,Reactome_2022.gmt,Synthesis Of PIPs At Late Endosome Membrane R-...,1/11,0.001049,0.006484,2993.571429,20536.174668,PIK3C2A,Reactome
30,Reactome_2022.gmt,Golgi Associated Vesicle Biogenesis R-HSA-432722,1/55,0.005244,0.012586,574.321101,3015.561294,PIK3C2A,Reactome
31,Reactome_2022.gmt,Synthesis Of PIPs At Plasma Membrane R-HSA-166...,1/51,0.004863,0.012586,620.049505,3302.483735,PIK3C2A,Reactome
32,Reactome_2022.gmt,PI Metabolism R-HSA-1483255,1/82,0.007818,0.013403,383.061350,1858.333183,PIK3C2A,Reactome
33,Reactome_2022.gmt,trans-Golgi Network Vesicle Budding R-HSA-199992,1/71,0.006770,0.013403,443.297872,2214.409031,PIK3C2A,Reactome
0,GO_Biological_Process_2021.gmt,vasoconstriction (GO:0042310),1/11,0.002279,0.016505,826.257143,5027.076195,PIK3C2A,GO_BP
1,GO_Biological_Process_2021.gmt,smooth muscle contraction (GO:0006939),1/14,0.002900,0.016505,642.511111,3754.320868,PIK3C2A,GO_BP
2,GO_Biological_Process_2021.gmt,clathrin coat assembly (GO:0048268),1/15,0.003106,0.016505,598.158621,3453.933068,PIK3C2A,GO_BP


In [34]:
import requests
import pandas as pd

# Mapping gene symbols → HGNC IDs
gene_to_hgnc = {
    "PIK3C2A": "HGNC:8977",
    "DNAJC27": "HGNC:30269",
    "IGSF10": "HGNC:29127"
}

def get_hpo_for_hgnc(hgnc_id):
    url = f"https://api.monarchinitiative.org/api/bioentity/{hgnc_id}/phenotypes"
    r = requests.get(url)
    if not r.ok:
        return []
    data = r.json()
    phenos = []
    for p in data.get("associations", []):
        phenos.append({
            "hgnc_id": hgnc_id,
            "hpo_id": p["object"]["id"],
            "hpo_label": p["object"]["label"],
            "source": p.get("source", "")
        })
    return phenos

all_pheno = []
for gene, hgnc in gene_to_hgnc.items():
    phenos = get_hpo_for_hgnc(hgnc)
    for ph in phenos:
        ph["gene"] = gene
    all_pheno.extend(phenos)

df = pd.DataFrame(all_pheno)
print(df)
df.to_csv("/Volumes/mac1/data/limb/HPO_monarch_results.csv", index=False)


Empty DataFrame
Columns: []
Index: []
